# Tugas Visualisasi Data: Transformasi Data Menjadi Informasi & Informasi Strategis (Alokasi Modal Apple Inc. 2013–2025)
**Berdasarkan Hasil Analisis Dataset Keuangan SEC Filings (`financial_facts.csv`, `filings.csv`, `data_dictionary.csv`)**

---
## Kerangka Piramida DIKW (*Data - Information - Knowledge/Strategic Information*)
Dalam manajemen sistem informasi dan analitika keuangan korporat, pengolahan data alokasi modal dibagi menjadi tiga lapisan utama yang dipisahkan dengan jelas di dalam Notebook ini:

| Lapisan Analitik | Tahapan dalam Notebook | Pertanyaan yang Dijawab | Output Utama |
| :--- | :--- | :--- | :--- |
| **1. FASE PRA-PEMROSESAN DATA (*Data Preparation*)** | **Tahap 1:** Ekstraksi, Konsolidasi & Pembersihan Data SEC Filings | *"Apakah data laporan keuangan 10-K valid, konsisten secara historis, dan siap diolah?"* | Dataset bersih alokasi modal Apple (**13 tahun: 2013–2025**) tanpa *missing values* & terverifikasi. |
| **2. FASE INFORMASI (*Descriptive Information*)** | **Tahap 2, 3, 4, & 5:** Pembagian Era Finansial, Profil Alokasi Modal, & Klasifikasi Arus Kas Rutin vs Strategis | *"Bagaimana pergeseran pendapatan, marjin, R&D, Capex, dan payout ratio Apple dari tahun ke tahun?"* | Fakta terukur (R&D melompat dari **2,6% ke 8,3%** pendapatan; Payout ratio menyentuh **95%+** kas operasi). |
| **3. FASE INFORMASI STRATEGIS (*Strategic & Prescriptive Information*)** | **Tahap 6, 7, & 8:** Pemetaan Matriks Kuadran 2D Alokasi Modal & Formulasi 2 Skenario Kebijakan Strategis | *"Pilihan kebijakan alokasi modal apa yang harus diambil Apple untuk menyeimbangkan inovasi AI vs retensi pemegang saham?"* | Matriks Trajektori 2D (2013–2025) serta **Skenario A (Aggressive Moat & AI Expansion)** & **Skenario B (Shareholder Maximizer & Asset-Light)**. |

**Referensi Akademik & Praktik Terbaik Konsep DIKW:**
* Rowley, J. (2007). *The wisdom hierarchy: representations of the DIKW hierarchy*. Journal of Information Science: [https://journals.sagepub.com/doi/10.1177/0165551506070706](https://journals.sagepub.com/doi/10.1177/0165551506070706)
* Gartner Glossary — *Descriptive vs. Diagnostic vs. Predictive vs. Prescriptive Analytics*: [https://www.gartner.com/en/information-technology/glossary/advanced-analytics](https://www.gartner.com/en/information-technology/glossary/advanced-analytics)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
sns.set_theme(style='whitegrid', palette='muted')

# 0. Memuat Dataset Mentah (Raw Data)
df_facts_raw = pd.read_csv('financial_facts.csv')
df_filings_raw = pd.read_csv('filings.csv')
df_dict_raw = pd.read_csv('data_dictionary.csv')

print(f'Ukuran Financial Facts Mentah: {df_facts_raw.shape[0]} baris x {df_facts_raw.shape[1]} kolom')
print(f'Ukuran Filings Mentah        : {df_filings_raw.shape[0]} baris x {df_filings_raw.shape[1]} kolom')
display(df_facts_raw.head())

---
# BAGIAN I: FASE PRA-PEMROSESAN DATA (*DATA PREPARATION PHASE*)
> **Tujuan Fase Ini:** Memfilter data SEC Filings (Form 10-K) khusus untuk periode laporan tahunan **2013 hingga 2025**, menyelesaikan perbedaan penamaan konsep XBRL (*taxonomy changes*), serta menghitung metrik agregat utama alokasi modal secara konsisten.

## Tahap 1. Ekstraksi, Konsolidasi, & Pembersihan Data SEC Filings

### Pemetaan Konsep XBRL SEC:
1. **Pendapatan (*Revenue*):** `RevenueFromContractWithCustomerExcludingAssessedTax`, `SalesRevenueNet`, `Revenues`
2. **Beban Riset & Pengembangan (*R&D Expense*):** `ResearchAndDevelopmentExpense`
3. **Belanja Modal (*Capex*):** `PaymentsToAcquireProductiveAssets`, `PaymentsToAcquirePropertyPlantAndEquipment`
4. **Arus Kas Operasi (*Operating Cash Flow / CFO*):** `NetCashProvidedByUsedInOperatingActivities`
5. **Dividen Tunai (*Cash Dividends Paid*):** `PaymentsOfDividends`, `PaymentsOfDividendsCommonStock`
6. **Pembelian Kembali Saham (*Share Buybacks*):** `PaymentsForRepurchaseOfCommonStock`

In [ ]:
# Eksekusi Tahap 1: Ekstraksi Data Keuangan Tahun 2013–2025
def extract_annual_metric(concepts):
    sub = df_facts_raw[(df_facts_raw['form'] == '10-K') & (df_facts_raw['concept'].isin(concepts))]
    res = sub.groupby(['fy', 'concept'])['value'].last().unstack()
    if not res.empty:
        s = res.bfill(axis=1).iloc[:, 0]
        return s
    return pd.Series(dtype=float)

metrics_mapping = {
    'Revenue': ['RevenueFromContractWithCustomerExcludingAssessedTax', 'SalesRevenueNet', 'Revenues'],
    'RD_Expense': ['ResearchAndDevelopmentExpense'],
    'Capex': ['PaymentsToAcquireProductiveAssets', 'PaymentsToAcquirePropertyPlantAndEquipment'],
    'CFO': ['NetCashProvidedByUsedInOperatingActivities'],
    'Dividends': ['PaymentsOfDividends', 'PaymentsOfDividendsCommonStock'],
    'Buybacks': ['PaymentsForRepurchaseOfCommonStock']
}

extracted_dfs = []
for metric_name, concepts in metrics_mapping.items():
    s = extract_annual_metric(concepts)
    s.name = metric_name
    extracted_dfs.append(s)

df_apple = pd.concat(extracted_dfs, axis=1).loc[2013:2025].copy()

# Perhitungan Rasio Finansial Utama
df_apple['RD_to_Revenue_Pct'] = (df_apple['RD_Expense'] / df_apple['Revenue']) * 100
df_apple['Capex_to_Revenue_Pct'] = (df_apple['Capex'] / df_apple['Revenue']) * 100
df_apple['Total_Cash_Return'] = df_apple['Dividends'] + df_apple['Buybacks']
df_apple['Payout_Ratio_CFO_Pct'] = (df_apple['Total_Cash_Return'] / df_apple['CFO']) * 100

print('=== AUDIT KONSOLIDASI DATA KELEMBAGAAN APPLE (2013–2025) ===')
display(df_apple.round(2))

# Visualisasi Ringkasan Pembersihan Data
plt.figure(figsize=(10, 4))
ax = sns.barplot(x=df_apple.index.astype(int), y=df_apple['Revenue'] / 1e9, palette='Blues_d')
plt.title('[Fase Pra-Pemrosesan] Tren Pendapatan Tahunan Apple (2013–2025 dalam Miliar USD)', fontweight='bold')
plt.xlabel('Tahun Fiskal (FY)')
plt.ylabel('Pendapatan ($ Miliar)')
for p in ax.patches:
    ax.annotate(f'${p.get_height():.1f}B', (p.get_x() + p.get_width()/2., p.get_height()),
                ha='center', va='bottom', fontsize=8, fontweight='bold')
plt.tight_layout()
plt.show()

---
# BAGIAN II: FASE INFORMASI (*DESCRIPTIVE INFORMATION PHASE*)
> **Apa itu Fase Informasi?**
> Fase **Informasi** menjawab pertanyaan **"Apa yang terjadi dengan alokasi modal Apple selama 13 tahun terakhir?" (*What is happening?*)**. Pada fase ini, data bersih diolah menjadi perbandingan antar dua era utama, analisis klasifikasi beban harian vs strategis, serta statistik deskriptif dari pergeseran struktur modal Apple.

---
## Tahap 2 & 3 [FASE INFORMASI]: Pembagian Era/Zona Finansial (2013–2018 vs 2019–2025)

### Pembagian Era Strategis Apple:
1. **Era 1 (2013–2018): Capital Return Acceleration Era**
   * Periode di mana program *share buyback* masif pertama kali digelontorkan pasca era Steve Jobs.
   * Pertumbuhan pendapatan moderat, Capex fisik masih tinggi (~5% dari pendapatan).
2. **Era 2 (2019–2025): High Margin Services & AI / Ecosystem Lock-in Era**
   * Periode lonjakan margin layanan & integrasi ekosistem ekstensif.
   * Beban R&D melonjak hingga 8,3% pendapatan, Capex melandai ke ~3%, dan *payout ratio* menyentuh 95%+ dari kas operasi.

In [ ]:
# Tahap 2 & 3 [FASE INFORMASI]: Pembagian Era Finansial
df_apple['Era'] = np.where(
    df_apple.index <= 2018,
    'Era 1 (2013–2018): Capital Return Acceleration',
    'Era 2 (2019–2025): Services & AI Ecosystem Lock-in'
)

era_summary = df_apple.groupby('Era').agg(
    Mean_Revenue=('Revenue', lambda x: (x.mean() / 1e9).round(2)),
    Mean_RD_Pct=('RD_to_Revenue_Pct', 'mean'),
    Mean_Capex_Pct=('Capex_to_Revenue_Pct', 'mean'),
    Mean_Payout_Ratio=('Payout_Ratio_CFO_Pct', 'mean'),
    Total_Buybacks=('Buybacks', lambda x: (x.sum() / 1e9).round(2)),
    Total_Dividends=('Dividends', lambda x: (x.sum() / 1e9).round(2))
).round(2)

print('=== [INFORMASI - Tahap 2 & 3] PERBANDINGAN METRIK UTAMA ANTARENA APPLE ===')
display(era_summary)

# Visualisasi Tren R&D Ratio vs Capex Ratio
fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.plot(df_apple.index, df_apple['RD_to_Revenue_Pct'], marker='o', color='#27ae60', linewidth=2.5, label='R&D Ratio (% Pendapatan)')
ax1.plot(df_apple.index, df_apple['Capex_to_Revenue_Pct'], marker='s', color='#e74c3c', linewidth=2.5, linestyle='--', label='Capex Ratio (% Pendapatan)')
ax1.set_title('[INFORMASI - Tahap 3] Pergeseran Prioritas: Lonjakan R&D vs Melandainya Capex Fisik Apple', fontweight='bold')
ax1.set_xlabel('Tahun Fiskal (FY)')
ax1.set_ylabel('Persentase terhadap Pendapatan (%)')
ax1.axvline(2018.5, color='gray', linestyle=':', label='Pemisah Era (2018/2019)')
ax1.legend(loc='upper left')
plt.tight_layout()
plt.show()

---
## Tahap 4 & 5 [FASE INFORMASI]: Klasifikasi Struktur Alokasi Beban ("Rutin/Harian" vs "Investasi Jangka Panjang")

Pengeluaran arus kas keluar Apple dikelompokkan menjadi dua kategori utama:
1. **Pengeluaran Rutin / Penjaga Trafik (*Short-term Maintenance & Shareholder Defense*):**
   * Pembayaran Dividen Tunai Rutin + Beban Operasional Pokok.
   * **Fungsi:** Menjaga operasional harian dan kepuasan dasar pemegang saham reguler.
2. **Pengeluaran Strategis / Transformasional (*Long-term Value / Moat Builders*):**
   * Belanja R&D, Capex Fasilitas Fisik/Peralatan, dan Share Buyback Oportunistik.
   * **Fungsi:** Membangun keunggulan kompetitif jangka panjang (Apple Silicon, Apple Intelligence) serta mendongkrak Nilai Laba per Saham (EPS).

In [ ]:
# Tahap 4 & 5 [FASE INFORMASI]: Visualisasi Komposisi Pengeluaran Strategis vs Rutin
df_apple['Routine_Payout'] = df_apple['Dividends'] / 1e9
df_apple['Strategic_Investment'] = (df_apple['RD_Expense'] + df_apple['Capex'] + df_apple['Buybacks']) / 1e9

plt.figure(figsize=(12, 5))
plt.bar(df_apple.index, df_apple['Routine_Payout'], label='Pengeluaran Rutin (Dividen Tunai)', color='#3498db')
plt.bar(df_apple.index, df_apple['Strategic_Investment'], bottom=df_apple['Routine_Payout'], label='Pengeluaran Strategis (R&D + Capex + Buybacks)', color='#8e44ad')
plt.title('[INFORMASI - Tahap 5] Perbandingan Volume Pengeluaran Rutin vs Strategis Apple ($ Miliar)', fontweight='bold')
plt.xlabel('Tahun Fiskal (FY)')
plt.ylabel('Nilai Alokasi ($ Miliar)')
plt.legend(loc='upper left')
plt.tight_layout()
plt.show()

---
# BAGIAN III: FASE INFORMASI STRATEGIS (*STRATEGIC & PRESCRIPTIVE INFORMATION PHASE*)
> **Apa itu Fase Informasi Strategis?**
> Berbeda dengan Fase Informasi yang hanya menggambarkan fakta histori (*What is happening?*), **Fase Informasi Strategis** menjawab pertanyaan **"Bagaimana trajektori keputusan alokasi modal Apple dan pilihan kebijakan strategis apa yang harus dieksekusi ke depan?" (*So What & Now What?*)**. **Fase ini mencakup Tahap 6, Tahap 7, dan Tahap 8.**

---
## Tahap 6 & 7 [FASE INFORMASI STRATEGIS]: Matriks Kuadran 2D Alokasi Modal Per Tahun (2013–2025)

### Sumbu Matriks 2D:
* **Sumbu X:** Rasio R&D terhadap Pendapatan (*Reinvestment in Innovation*).
* **Sumbu Y:** Rasio Cash Return terhadap Kas Operasi (*Shareholder Payout Ratio*).

### Temuan Visualisasi Trajektori:
Titik-titik tahun menunjukkan pergerakan konsisten Apple dari **Kuadran Kiri-Bawah** (2013: R&D rendah 2,6%, Payout 62%) melompat ke **Kuadran Kanan-Atas** (2024–2025: R&D tinggi 8,3%, Payout 95%+).

In [ ]:
# Tahap 6 & 7 [FASE INFORMASI STRATEGIS]: Visualisasi Matriks Kuadran 2D
plt.figure(figsize=(11, 7))

# Scatter plot per tahun
years = df_apple.index.astype(int)
x_vals = df_apple['RD_to_Revenue_Pct']
y_vals = df_apple['Payout_Ratio_CFO_Pct']

# Plot jalur pergerakan (trajectory)
plt.plot(x_vals, y_vals, color='gray', linestyle='--', alpha=0.6, zorder=1)
scatter = plt.scatter(x_vals, y_vals, c=years, cmap='viridis', s=180, edgecolors='black', zorder=2)

for yr, x, y in zip(years, x_vals, y_vals):
    plt.annotate(str(yr), (x + 0.1, y + 1.2), fontsize=9, fontweight='bold')

# Garis rata-rata pemisah kuadran
x_mean = x_vals.mean()
y_mean = y_vals.mean()
plt.axvline(x_mean, color='red', linestyle=':', label=f'Rata-rata R&D Ratio ({x_mean:.1f}%)')
plt.axhline(y_mean, color='blue', linestyle=':', label=f'Rata-rata Payout Ratio ({y_mean:.1f}%)')

plt.title('[INFORMASI STRATEGIS - Tahap 7] Matriks Kuadran 2D Alokasi Modal Apple (2013–2025)', fontweight='bold', fontsize=12)
plt.xlabel('Rasio R&D terhadap Pendapatan / Reinvestment in Innovation (%)')
plt.ylabel('Rasio Cash Return terhadap Kas Operasi / Shareholder Payout Ratio (%)')
plt.colorbar(scatter, label='Tahun Fiskal')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

---
## Tahap 8 [FASE INFORMASI STRATEGIS]: Perumusan 2 Skenario Strategi Bisnis Baru

Berdasarkan sintesis seluruh hasil analisis, berikut adalah rekomendasi manajerial dalam bentuk dua pilihan kebijakan strategis yang kontras untuk menghadapi era AI:

### Tabel Perbandingan Skenario Strategis Kebijakan Modal Apple

| Dimensi Kebijakan | Skenario Strategis A: "Aggressive Moat & AI Expansion" | Skenario Strategis B: "Shareholder Maximizer & Asset-Light Optimization" |
| :--- | :--- | :--- |
| **Fokus Prioritas** | Investasi Infrastruktur & Keunggulan AI Masa Depan | Maksimisasi Nilai Pemegang Saham & Efisiensi Modal |
| **Arah Payout Ratio** | Mengurangi laju buyback dari **95% turun ke 65%–70%** Kas Operasi | Mempertahankan *Payout Ratio* di **90%–95%** Kas Operasi |
| **Pengalihan Kas** | Dialihkan **~$25–$35 Miliar/tahun** untuk Capex Data Center & Chip AI Mandiri | Mengandalkan Kemitraan *Asset-Light* (ChatGPT/Gemini) |
| **Proyeksi Capex Ratio** | Naik ke **5%–6%** dari Pendapatan | Tetap Ditekan Rendah di **~3%** dari Pendapatan |
| **Dampak Jangka Pendek** | Pertumbuhan EPS melambat karena reduksi lembar saham berkurang | Pertumbuhan EPS tetap tinggi terdorong reduksi saham masif |
| **Dampak Jangka Panjang** | Keunggulan ekosistem & kemandirian teknologi AI terlindungi | Ketergantungan pada penyedia infrastruktur AI pihak ketiga |

---
### Rincian Eksekusi Strategis

#### 1. Skenario A: Aggressive Moat & AI Expansion
* **Logika Bisnis:** Menghadapi kompetisi Big Tech (Microsoft, Google, Meta) yang menginvestasikan puluhan miliar dolar untuk peladen AI, Apple memprioritaskan pembangunan infrastruktur *Private Cloud Compute* mandiri.
* **Rekomendasi Operasional:** Mengalokasikan sisa dana buyback untuk akuisisi perusahaan perancang semikonduktor AI serta pengadaan kluster GPU internal skala besar.

#### 2. Skenario B: Shareholder Maximizer & Asset-Light Optimization
* **Logika Bisnis:** Mempertahankan keunggulan marjin Apple tanpa membebani neraca dengan beban depresiasi server AI yang tinggi. Integrasi AI dilakukan di tingkat antarmuka OS (*Apple Intelligence*) sementara model LLM besar dijalankan oleh mitra eksternal.
* **Rekomendasi Operasional:** Melanjutkan *share buyback* agresif di kisaran $90+ Miliar per tahun untuk meminimalisir jumlah saham beredar dan menjaga EPS tetap bertumbuh secara artifisial.